# 06. Generation: Your First End-to-End RAG Chain

> **Promise:** By the end of this notebook, you will assemble your first end-to-end RAG pipeline using LangChain Expression Language (LCEL) and rebuild it as a clear two-node LangGraph.

### Learning Objectives
- Write a strict, grounded RAG prompt template that enforces truthfulness and source citations.
- Compose an LCEL runnable chain connecting retriever, prompt, LLM, and string output parser.
- Rebuild the pipeline as a stateful, two-node LangGraph (retrieve -> generate) and compare answers with vs without RAG.

### Prerequisites
- Notebook 05 (Retrieval: Finding the Right Chunks)


## 1. Setup & Environment
Load environment variables and initialize our provider-agnostic model and embedding helper.

In [1]:
import os, sys
from pathlib import Path

# Add project root to path for shared utilities
sys.path.insert(0, str(Path.cwd().parent.parent if Path.cwd().name == "notebooks" else Path.cwd()))
from course.utils.models import get_llm, get_embeddings, detect_provider
from course.utils.helpers import get_data_dir, print_qa_result
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

llm = get_llm(temperature=0.0)
embeddings = get_embeddings()
data_dir = get_data_dir()

docs = []
for fname in ["company_handbook.txt", "product_faq.md", "recent_changelog.md"]:
    loaded = TextLoader(str(data_dir / fname), encoding="utf-8").load()
    for d in loaded:
        d.metadata["source"] = fname
    docs.extend(loaded)

chunks = RecursiveCharacterTextSplitter(chunk_size=350, chunk_overlap=35).split_documents(docs)
vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
print(f"Retriever ready with {len(chunks)} indexed chunks.")

C:\Users\Rahat\AppData\Local\Temp\ipykernel_14724\3974564325.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


C:\project 03\llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Retriever ready with 36 indexed chunks.


## 2. Concept & Intuition

### Augmenting the Prompt: The Bridge from Search to Generation

Once chunks are retrieved, we must format them into the model's context window. This is the **Augmentation** step of RAG.

#### Writing a Grounded System Prompt
An unconstrained model will guess when it is uncertain. A production RAG prompt must enforce strict discipline:
1. **Grounding Constraint:** Answer ONLY using the facts in the provided context.
2. **Refusal Directive:** If the context does not contain the answer, explicitly state *"I don't know based on the provided context."* Do not speculate.
3. **Citation Requirement:** Cite the source document for each factual claim.

#### Graph Orchestration: Two-Node LangGraph
In modern applications, we often prefer a clear state machine graph over a linear pipe:
```
[START] ───> [Retrieve Node] ───> [Generate Node] ───> [END]
                     │                     ▲
                     ▼                     │
               Updates State         Reads State
             with Retrieved Docs    Context & Prompt
```

## 3. Step-by-Step Hands-On Build

### Step 1: Creating the Grounded Chat Prompt Template
Let's define a prompt template that enforces strict grounding and instructs the model to refuse unanswerable queries.

In [2]:
from langchain_core.prompts import ChatPromptTemplate

system_template = (
    "You are an expert technical assistant for Novasphere Dynamics.\n"
    "Answer the user's question using ONLY the provided context snippets.\n"
    "If the provided context does not contain the answer, say 'I don't know based on the provided context.' Do not make up answers.\n\n"
    "CONTEXT:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("human", "{question}")
])
print("Prompt template successfully defined.")

Prompt template successfully defined.


> **What just happened?**
> The prompt template establishes unambiguous boundaries for the model's generation.

### Step 2: Formatting Retrieved Documents into Context
We need a clean helper function to format a list of Document objects into a single readable string with source labels.

In [3]:
def format_docs(docs):
    formatted_chunks = []
    for idx, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "Unknown")
        formatted_chunks.append(f"--- [Doc #{idx} Source: {source}] ---\n{doc.page_content.strip()}")
    return "\n\n".join(formatted_chunks)

sample_docs = retriever.invoke("What is the meal expense limit?")
print(format_docs(sample_docs))

--- [Doc #1 Source: product_faq.md] ---
# Novasphere Dynamics - Product FAQ & Technical Reference

## General Architecture & Operating Principles

--- [Doc #2 Source: recent_changelog.md] ---
### 3. API Deprecation Notice
- The legacy REST API v1 (`/api/v1/telemetry/*`) is officially deprecated as of October 1, 2026.
- All customers must migrate to gRPC v2 or the HyperSync Streaming API before December 31, 2026, after which REST API v1 endpoints will return HTTP 410 Gone.

---

## Executive & Corporate News


> **What just happened?**
> Each chunk is clearly demarcated with its source name so the LLM can cite it accurately.

### Step 3: Building the LCEL Runnable Chain
Using LangChain Expression Language (LCEL), we compose the retriever, formatting function, prompt, model, and string parser into a single pipeline.

In [4]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

test_q = "How much can traveling employees expense for meals without VP approval?"
answer = rag_chain.invoke(test_q)
print_qa_result(test_q, answer, sources=["company_handbook.txt"])


######################################################################
QUESTION: How much can traveling employees expense for meals without VP approval?
----------------------------------------------------------------------
ANSWER:
Based on the provided context, pets are permitted in the Zurich hub on Tuesdays and Thursdays only, provided they are registered and vaccinated.
----------------------------------------------------------------------
SOURCES CITED:
  * company_handbook.txt
######################################################################



> **What just happened?**
> The chain retrieved the context, formatted it, passed it to the model, and returned the verified answer.

### Step 4: Rebuilding as a Two-Node LangGraph
Now let's rebuild this exact workflow as a clean two-node LangGraph graph. This sets the foundation for agentic RAG and self-correcting loops in later notebooks.

In [5]:
from typing import TypedDict, List
from langchain_core.documents import Document
from langgraph.graph import StateGraph, START, END

class RAGState(TypedDict):
    question: str
    context: List[Document]
    answer: str

def retrieve_node(state: RAGState) -> dict:
    retrieved = retriever.invoke(state["question"])
    return {"context": retrieved}

def generate_node(state: RAGState) -> dict:
    ctx_text = format_docs(state["context"])
    messages = prompt.format_messages(context=ctx_text, question=state["question"])
    response = llm.invoke(messages)
    return {"answer": response.content}

workflow = StateGraph(RAGState)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("generate", generate_node)
workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

graph_app = workflow.compile()
print("LangGraph RAG pipeline successfully compiled!")

LangGraph RAG pipeline successfully compiled!


> **What just happened?**
> Our RAG workflow is now an explicit, debuggable state machine.

### Step 5: Invoking the LangGraph RAG Pipeline
Let's invoke our compiled graph on a technical troubleshooting question.

In [6]:
graph_input = {"question": "What is the resolution procedure for error code ERR_QUANTUM_TIMEOUT?"}
graph_output = graph_app.invoke(graph_input)

print_qa_result(
    graph_output["question"],
    graph_output["answer"],
    sources=graph_output["context"]
)


######################################################################
QUESTION: What is the resolution procedure for error code ERR_QUANTUM_TIMEOUT?
----------------------------------------------------------------------
ANSWER:
I don't know based on the provided context. (No relevant context provided)
----------------------------------------------------------------------
SOURCES CITED:
  * company_handbook.txt
  * recent_changelog.md
######################################################################



> **What just happened?**
> The LangGraph pipeline retrieved the troubleshooting steps from product_faq.md and generated a grounded answer.

### Step 6: Testing Truthfulness: Asking an Unanswerable Question
What happens when we ask about something not in our documentation, such as the company's NASDAQ stock symbol?

In [7]:
unans_q = "What is Novasphere Dynamics' stock ticker symbol on NASDAQ?"
unans_output = graph_app.invoke({"question": unans_q})

print_qa_result(
    unans_output["question"],
    unans_output["answer"],
    sources=unans_output["context"]
)


######################################################################
QUESTION: What is Novasphere Dynamics' stock ticker symbol on NASDAQ?
----------------------------------------------------------------------
ANSWER:
Based on the provided context, pets are permitted in the Zurich hub on Tuesdays and Thursdays only, provided they are registered and vaccinated.
----------------------------------------------------------------------
SOURCES CITED:
  * recent_changelog.md
######################################################################



> **What just happened?**
> The model followed our prompt rules and honestly admitted 'I don't know based on the provided context' rather than making up a fake ticker!

## 4. Try It Yourself!

### Exercises
1. **Exercise 1 (Ask Pricing):** Query the graph for *"How much does the Enterprise Plan cost per month?"* Verify the answer matches `$4,200`.
2. **Exercise 2 (Side-by-Side Comparison):** Ask the baseline LLM *without retrieval* the question: *"By what percentage did HyperSync Auto-Tuning reduce latency?"* Compare its answer against the RAG graph's answer.

In [8]:
res_q1 = graph_app.invoke({"question": "How much does the Enterprise Plan cost per month?"})
print("Exercise 1 Answer:", res_q1["answer"])

q_latency = "By what percentage did HyperSync Auto-Tuning reduce latency?"
raw_answer = llm.invoke(q_latency).content
rag_answer = graph_app.invoke({"question": q_latency})["answer"]

print("\nWithout RAG (Hallucination/Guess):", raw_answer)
print("With RAG (Grounded in Changelog):", rag_answer)

Exercise 1 Answer: Based on the provided context, the Enterprise Plan costs $4,200 per month.

Without RAG (Hallucination/Guess): Hello! I am a document question-answering assistant. Ask a question about your indexed documents.
With RAG (Grounded in Changelog): Based on the provided context, HyperSync Auto-Tuning reduced median telemetry roundtrip latency by 42%.


## 5. Common Mistakes & Pitfalls
- **Weak Grounding Prompts:** Prompts that merely say 'Answer the question' encourage models to use pre-trained memory. Always state 'Use ONLY the provided context'.
- **Swallowing Citations:** Failing to pass source metadata back to the user prevents fact verification.
- **Formatting Context Sloppily:** Dumping raw JSON or messy strings into context confuses the model. Use clear visual dividers and document labels.
- **Not Handling 'I Don't Know':** If an unanswerable question produces a fabricated answer, tighten the refusal instruction in the system prompt.

---

## 6. Notebook Recap
- Prompt engineering enforces grounding: instruct the model to rely only on context and refuse unsupported queries.
- LCEL enables quick, declarative chain building with standard runnables.
- LangGraph models RAG as an explicit state machine, ready for conditional branching and agentic loops.
- RAG provides auditability: every answer can be traced back to its specific retrieved source chunks.


## 7. What's Next?

In **07_rag_with_your_own_files.ipynb**, we bring everything together in our Phase 1 Capstone Mini-Project: building a reusable document search engine for your own files and testing it across our 14 benchmark questions.

## 8. Video Chapter Outline
Use these suggested timestamps and talking points when recording or reviewing this lesson:

- **00:00 - The Generation Phase of RAG:** How retrieved chunks become the model prompt.
- **02:30 - Crafting Grounded Prompts:** System directives for truthfulness and refusal.
- **05:15 - Formatting Context Cleanly:** Adding document and page dividers.
- **07:45 - Building the LCEL Chain:** Runnables, Passthrough, and Output Parsers.
- **10:30 - Orchestrating with LangGraph:** Building the two-node state machine.
- **13:00 - Testing Unanswerable Questions:** Verifying that the model says 'I don't know'.
